Fig3C_ripple ablate

In [ ]:
# Data for this notebook: ../csv/FigS3_B_S3C_S3D_ripple_ablate/  (exported from the lab data by
# ../analysis_scripts/make_figure_csvs.py; the original loading code is kept below as comments)
import os, glob, pickle, re, sys   # standard imports (some notebooks relied on them being loaded already)
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the lab's `criticality` package, shipped there
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'FigS3_B_S3C_S3D_ripple_ablate'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """Read one of this notebook's CSVs; round_trip parsing gives back exactly the numbers written."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match (works if those faces exist on your system)
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
# ---------------------------------------
# 1) Load all PKL files from out_dir
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import mpmath as mp
from scipy.stats import norm

import numpy as np

def p_str_from_p(p, digits=3):
    p = float(p)
    if not np.isfinite(p):
        return "nan"
    if p == 0.0:
        return "p<5e-324"   # can't recover exact value from 0.0
    return f"{p:.{digits}e}"

# ---------------------------------------------------------
# 1) Load all PKLs with filename column attached
# ---------------------------------------------------------
out_dir = Path(r"D:\Documents\GitHub\Sleep_criticaltiy\data\ripple_barr\ripple_ablate")

# ---- data: read from CSV (replaces the loading code below) ----
all_dfs = []
ca1_all_df = read_csv('d2_ripple_ablated_windows.csv')
# ---- original loading code (needs the lab servers), kept for reference ----
# all_dfs = []
# for p in out_dir.glob("*.pkl"):
#     try:
#         df = pd.read_pickle(p)
#         df["file_id"] = p.name   # <---- add filename to identify file
#         all_dfs.append(df)
#         print(f"Loaded: {p.name}, n={len(df)}")
#     except Exception as e:
#         print(f"!! Failed to load {p}: {e}")
# 
# if not all_dfs:
#     raise RuntimeError("No PKL files found")
# 
# ca1_all_df = pd.concat(all_dfs, ignore_index=True)
print("Total rows:", len(ca1_all_df))


# ---------------------------------------------------------
# 2) Z-score d² WITHIN each file_id
# ---------------------------------------------------------
def zscore(x):
    return (x - x.mean()) / x.std(ddof=0)

ca1_all_df["d2_z"] = (
    ca1_all_df.groupby("file_id")["d2"]
              .transform(zscore)
)

print("Done z-scoring within each file.")



 
# ---------------------------------------------------------
# 3) Restrict to NREM
# ---------------------------------------------------------
nrem = ca1_all_df[ca1_all_df["state"] == "NREM"].copy()
nrem = nrem[np.isfinite(nrem["d2_z"])]

epoch_order = ["presleep", "midsleep", "postsleep"]
epoch_colors = {
    "presleep": "#4c72b0",
    "midsleep": "#55a868",
    "postsleep": "#dd8452",
}
# =========================================================
# 4) Plot 1: d²_z vs ripple tiles (6 tiles: mean ± SEM)
# =========================================================
# =========================================================
# 4) Cross-file SEM for ripple tiles (PER-FILE SEXTILES)
# =========================================================

rate_col = "ripple_rate_30s"

nrem_rt = nrem[np.isfinite(nrem[rate_col])].copy()
NTILES = 6

# assign tiles (per-file quantiles)
nrem_rt["ripple_tile"] = (
    nrem_rt.groupby("file_id")[rate_col]
           .transform(lambda x: pd.qcut(x, q=NTILES, labels=False, duplicates="drop"))
)

# ---- compute tile mean *per file* ----
per_file_tile = (
    nrem_rt
    .groupby(["file_id", "ripple_tile"])["d2_z"]
    .mean()
    .reset_index()
)

# ---- now compute mean & SEM across files ----
tile_stats = (
    per_file_tile
    .groupby("ripple_tile")["d2_z"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

tile_stats["sem"] = tile_stats["std"] / np.sqrt(tile_stats["count"])

print(tile_stats)

# Plot mean ± SEM
fig1, ax1 = plt.subplots(figsize=(5, 4))  # 6 inches wide, 4 inches tall



tile_x = tile_stats["ripple_tile"] + 1   # shift so tiles = 1..6

ax1.errorbar(
    tile_x,
    tile_stats["mean"],
    yerr=tile_stats["sem"],
    color="tab:red",
    marker="o",
    linestyle="-", capsize=4, linewidth=2
)

ax1.set_xlabel("Ripple rate quantiles (per-file)")
ax1.set_ylabel("d2 distance (z)")
ax1.set_title("NREM: z-d² after ripple ablation\n(cross-file mean ± SEM)")
ax1.set_xticks([1,2,3,4,5,6])
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
save_fig_path=r"../figure_output\S3_C_ripple_ablate_d2z_vs_ripplerate_perfile_sextile.svg"
fig1.savefig(save_fig_path)
plt.tight_layout()
# =========================================================
# 6) Linear trend test across ripple tiles
# =========================================================
import scipy.stats as stats

# Prepare vectors
x = per_file_tile["ripple_tile"].astype(float)
y = per_file_tile["d2_z"].astype(float)

# -------- Linear regression (parametric) --------
lr = stats.linregress(x, y)
p_str = p_str_from_p(lr.pvalue)
print("\n===== Linear Trend Test (Parametric) =====")
print(f"Slope = {lr.slope:.5f}")
print(f"Intercept = {lr.intercept:.5f}")
print(f"r = {lr.rvalue:.5f}")
print(f"p-value = {p_str}")
print(f"Std error = {lr.stderr:.5f}")

# -------- Spearman rank correlation (non-parametric) --------
rho, p_s = stats.spearmanr(x, y)

print("\n===== Linear Trend Test (Spearman) =====")
print(f"Spearman rho = {rho:.5f}")
print(f"p-value = {p_str_from_p(p_s)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False
mpl.rcParams["axes.linewidth"] = 1.5
mpl.rcParams["xtick.major.width"] = 1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5

def _despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

# -------------------------
# Settings
# -------------------------
plot_epochs = ["presleep", "midsleep"]
xtick_labels = ["presleep", "postsleep"]   # display label remap midsleep->postsleep
epoch_color_map = {
    "presleep": epoch_colors["presleep"],
    "midsleep": epoch_colors["postsleep"],  # intentional remap
}

# REQUIRED: column that identifies session
# change "session" -> whatever you actually have, e.g. "file", "basepath", "animal_day"
SESSION_COL = "file_id"

# Choose your within-session summary statistic:
AGG = "mean"  # "mean" or "median"

# -------------------------
# 1) Reduce to one number per session per epoch
# -------------------------
df = nrem.loc[nrem["sleep_epoch"].isin(plot_epochs), [SESSION_COL, "sleep_epoch", "d2"]].dropna().copy()

if AGG == "mean":
    sess = (df.groupby([SESSION_COL, "sleep_epoch"])["d2"]
              .mean()
              .rename("d2_z_sess")
              .reset_index())
else:
    sess = (df.groupby([SESSION_COL, "sleep_epoch"])["d2"]
              .median()
              .rename("d2_z_sess")
              .reset_index())

# Pivot to wide (paired)
wide = sess.pivot(index=SESSION_COL, columns="sleep_epoch", values="d2_z_sess")

# Keep only sessions with both epochs
wide = wide.dropna(subset=plot_epochs)

pre = wide["presleep"].to_numpy()
mid = wide["midsleep"].to_numpy()
n_pair = len(wide)

print(f"Paired sessions: {n_pair}")

# -------------------------
# 2) Plot: paired lines + boxplot overlay
# -------------------------
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# wide has index = session, columns = presleep / midsleep
# wide = wide.dropna(subset=["presleep","midsleep"])

long = (
    wide
    .reset_index()
    .melt(id_vars=SESSION_COL,
          value_vars=["presleep", "midsleep"],
          var_name="sleep_epoch",
          value_name="d2")
)

# remap midsleep -> postsleep for display
long["sleep_epoch_plot"] = long["sleep_epoch"].replace({"midsleep": "postsleep"})

fig, ax = plt.subplots(figsize=(3.6, 4))
epoch_colors={
    "presleep": epoch_color_map["presleep"],
    "postsleep": epoch_color_map["midsleep"]  # intentional remap
}
palette = {
    "presleep": epoch_colors["presleep"],
    "postsleep": epoch_colors["postsleep"]
}

# --- boxplot ---
sns.boxplot(
    data=long,
    x="sleep_epoch_plot",
    y="d2",
    order=["presleep", "postsleep"],
    palette=palette,
    width=0.6,
    showfliers=False,
    linewidth=2,
    ax=ax,
    zorder=1
)

# --- jittered points ---
sns.stripplot(
    data=long,
    x="sleep_epoch_plot",
    y="d2",
    order=["presleep", "postsleep"],
    color="k",
    size=6,
    jitter=0,
    alpha=0.75,
    ax=ax,
    zorder=2
)

# --- paired lines ---
sns.lineplot(
    data=long,
    x="sleep_epoch_plot",
    y="d2",
    units=SESSION_COL,    estimator=None,
    sort=False,
    color="gray",
    alpha=0.5,
    linewidth=1.2,
    legend=False,
    ax=ax,
    zorder=3
)

# formatting
ax.set_xlabel("Sleep epoch (NREM)")
ax.set_ylabel("d² distance")
_despine(ax)

plt.tight_layout()
save_path = r"../figure_output\S2_B_ripple_ablate_d2z_vs_sleepepoch_paired.svg"
plt.savefig(save_path,dpi=300,bbox_inches="tight")
plt.show()

import pandas as pd
import statsmodels.formula.api as smf

# long must contain: d2, sleep_epoch_plot (presleep/postsleep), file_id, animal
# If 'animal' isn't in long yet, merge it in from nrem before this step.

lmm_df = long.copy()
lmm_df["animal"] = lmm_df["file_id"].str.extract(r"^([A-Za-z]+[0-9]+)", expand=False)
# Ensure the epoch is categorical with presleep as reference
lmm_df["sleep_epoch_plot"] = pd.Categorical(
    lmm_df["sleep_epoch_plot"],
    categories=["presleep", "postsleep"],
    ordered=True
)

# Paired LMM:
# - groups = animal (random intercept per animal)
# - vc_formula adds a random intercept per session (file_id),
#   which pairs presleep/postsleep within each file_id
model = smf.mixedlm(
    "d2 ~ C(sleep_epoch_plot)",              # fixed effect: epoch (post vs pre)
    data=lmm_df,
    groups=lmm_df["animal"],                 # random intercept: animal
    vc_formula={"session": "0 + C(file_id)"} # random intercept: session/file_id
)

res = model.fit(reml=False, method="lbfgs")
print(res.summary())

# Extract the p-value for postsleep vs presleep
term = "C(sleep_epoch_plot)[T.postsleep]"
p = res.pvalues[term]
beta = res.params[term]
se = res.bse[term]
print(f"post vs pre: beta={beta:.4g} ± {se:.4g}, p={p:.4g}")

In [ ]:
ca1_all_df

In [ ]:
ca1_all_df

In [ ]:
# Z-score ISI_entropy per session
def zscore(x):
    return (x - x.mean()) / x.std(ddof=0)

ca1_all_df["isi_entropy_z"] = (
    ca1_all_df.groupby("file_id")["isi_entropy"]
              .transform(zscore)
)

# =========================================================
# ISI entropy vs ripple sextile (per-file quantiles)
# =========================================================

rate_col = "ripple_rate_30s"
nrem=ca1_all_df[ca1_all_df["state"] == "NREM"].copy()
nrem_isi = nrem[np.isfinite(nrem[rate_col]) & np.isfinite(nrem["isi_entropy_z"])].copy()
NTILES = 6

# assign tiles (per-file quantiles)
nrem_isi["ripple_tile"] = (
    nrem_isi.groupby("file_id")[rate_col]
            .transform(lambda x: pd.qcut(x, q=NTILES, labels=False, duplicates="drop"))
)

# ---- compute tile mean *per file* ----
per_file_tile_isi = (
    nrem_isi
    .groupby(["file_id", "ripple_tile"])["isi_entropy_z"]
    .mean()
    .reset_index()
)

# ---- now compute mean & SEM across files ----
tile_stats_isi = (
    per_file_tile_isi
    .groupby("ripple_tile")["isi_entropy_z"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

tile_stats_isi["sem"] = tile_stats_isi["std"] / np.sqrt(tile_stats_isi["count"])

print(tile_stats_isi)

# Plot mean ± SEM
fig_isi, ax_isi = plt.subplots(figsize=(5, 4))

tile_x = tile_stats_isi["ripple_tile"] + 1   # shift so tiles = 1..6

ax_isi.errorbar(
    tile_x,
    tile_stats_isi["mean"],
    yerr=tile_stats_isi["sem"],
    color="tab:red",
    marker="o",
    linestyle="-", capsize=4, linewidth=2
)

ax_isi.set_xlabel("Ripple rate quantiles (per-file)")
ax_isi.set_ylabel("ISI entropy (z)")
ax_isi.set_title("NREM: ISI entropy vs ripple rate\n(cross-file mean ± SEM)")
ax_isi.set_xticks([1,2,3,4,5,6])
ax_isi.spines["top"].set_visible(False)
ax_isi.spines["right"].set_visible(False)

save_fig_path_isi = r"../figure_output\S3_D_ripple_ablate_isi_entropy_vs_ripplerate_perfile_sextile.svg"
fig_isi.savefig(save_fig_path_isi)
plt.tight_layout()
plt.show()

# =========================================================
# Linear trend test for ISI entropy
# =========================================================
import scipy.stats as stats

x_isi = per_file_tile_isi["ripple_tile"].astype(float)
y_isi = per_file_tile_isi["isi_entropy_z"].astype(float)

# Linear regression
lr_isi = stats.linregress(x_isi, y_isi)

print("\n===== ISI Entropy Linear Trend Test (Parametric) =====")
print(f"Slope = {lr_isi.slope:.5f}")
print(f"Intercept = {lr_isi.intercept:.5f}")
print(f"r = {lr_isi.rvalue:.5f}")
p_str_isi = p_str_from_p(lr_isi.pvalue)
print(f"p-value = {p_str_isi}")
print(f"Std error = {lr_isi.stderr:.5f}")

# Spearman
rho_isi, p_s_isi = stats.spearmanr(x_isi, y_isi)

print("\n===== ISI Entropy Linear Trend Test (Spearman) =====")
print(f"Spearman rho = {rho_isi:.5f}")
print(f"p-value = {p_str_from_p(p_s_isi)}")